
# S&P 500 Risk Dashboard

## Current quick-read metrics
- SPY vs 200D
- Breadth > 200D
- HY OAS
- SPY Drawdown
- VIX Level
- VIX Contango
- TNX Momentum Z
- Composite Risk Score

## Dashboard sections
1. KPI cards
2. Market state panel
3. VIX term structure (last 10 days)
4. Yield term structure (last 10 days)
5. Sector summary table
6. Quick read table


In [1]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML
from pandas_datareader import data as web
import ipywidgets as widgets

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [2]:

MAIN_TICKER = "SPY"
SECTOR_TICKERS = ["XLB", "XLC", "XLE", "XLF", "XLI", "XLK", "XLP", "XLRE", "XLU", "XLV", "XLY"]
VIX_TICKERS = ["^VIX", "^VIX3M", "^VIX6M"]
RATE_TICKERS = ["^TNX"]
ALL_TICKERS = [MAIN_TICKER] + SECTOR_TICKERS + VIX_TICKERS + RATE_TICKERS
FRED_SERIES = {"DGS1MO": "1M", "DGS3MO": "3M", "DGS6MO": "6M", "DGS1": "1Y", "DGS2": "2Y", "DGS3": "3Y", "DGS5": "5Y", "DGS7": "7Y", "DGS10": "10Y", "DGS20": "20Y", "DGS30": "30Y", "BAMLH0A0HYM2": "HY_OAS"}
TREASURY_ORDER = ["1M", "3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
TNX_MOMENTUM_WINDOW = 20
TNX_ZSCORE_WINDOW = 252


In [3]:

def download_yahoo_close(tickers, start_date):
    raw_data = yf.download(tickers, start=start_date, auto_adjust=True, progress=False, group_by="ticker")
    close_data = pd.DataFrame(index=raw_data.index)
    if isinstance(raw_data.columns, pd.MultiIndex):
        for ticker in tickers:
            if (ticker, "Close") in raw_data.columns:
                close_data[ticker] = raw_data[(ticker, "Close")]
    else:
        close_data[tickers[0]] = raw_data["Close"]
    return close_data.sort_index().dropna(how="all")

def download_fred_series(series_map, start_date):
    fred_data = pd.DataFrame()
    for fred_code, output_name in series_map.items():
        try:
            fred_data[output_name] = web.DataReader(fred_code, "fred", start_date)[fred_code]
        except Exception as error:
            print(f"Failed to load {fred_code}: {error}")
    return fred_data.sort_index()

def compute_drawdown(price_series):
    return price_series / price_series.cummax() - 1

def pct_above_moving_average(price_frame, moving_average_window):
    return (price_frame > price_frame.rolling(moving_average_window).mean()).mean(axis=1) * 100

def compute_dashboard_data(start_date="2010-01-01"):
    prices = download_yahoo_close(ALL_TICKERS, start_date)
    fred = download_fred_series(FRED_SERIES, start_date)
    data = prices.join(fred, how="left").ffill()
    data["SPY_200D"] = data[MAIN_TICKER].rolling(200).mean()
    data["SPY_drawdown"] = compute_drawdown(data[MAIN_TICKER])
    data["Breadth_above_200D"] = pct_above_moving_average(data[SECTOR_TICKERS], 200)
    data["10Y_minus_2Y"] = data["10Y"] - data["2Y"]
    data["VIX_contango_pct"] = (data["^VIX3M"] - data["^VIX"]) / data["^VIX"] * 100
    data["TNX_momentum"] = data["^TNX"] - data["^TNX"].shift(TNX_MOMENTUM_WINDOW)
    data["TNX_momentum_z"] = (data["TNX_momentum"] - data["TNX_momentum"].rolling(TNX_ZSCORE_WINDOW).mean()) / data["TNX_momentum"].rolling(TNX_ZSCORE_WINDOW).std()

    risk_components = pd.DataFrame(index=data.index)
    risk_components["Trend"] = (data[MAIN_TICKER] < data["SPY_200D"]).astype(float)
    risk_components["Breadth"] = (data["Breadth_above_200D"] < 45).astype(float)
    risk_components["Credit"] = (data["HY_OAS"] > data["HY_OAS"].rolling(252).quantile(0.75)).astype(float)
    risk_components["Drawdown"] = (data["SPY_drawdown"] < -0.08).astype(float)
    risk_components["Curve"] = (data["10Y_minus_2Y"] < 0).astype(float)
    risk_components["VIXLevel"] = (data["^VIX"] > 25).astype(float)
    risk_components["VIXContango"] = (data["VIX_contango_pct"] < 0).astype(float)
    risk_components["TNXMomentum"] = (data["TNX_momentum_z"].abs() > 1).astype(float)
    data["Composite_risk_score"] = risk_components.mean(axis=1) * 100

    sector_return_table = pd.DataFrame(index=SECTOR_TICKERS)
    for label, window in {"1D": 1, "1W": 5, "1M": 21, "3M": 63}.items():
        sector_return_table[label] = data[SECTOR_TICKERS].pct_change(window).iloc[-1] * 100
    sector_return_table["Above_200D"] = (data[SECTOR_TICKERS].iloc[-1] > data[SECTOR_TICKERS].rolling(200).mean().iloc[-1]).astype(int)
    sector_return_table = sector_return_table.sort_values("1M", ascending=False)
    return data, risk_components, sector_return_table


In [4]:

def make_kpi_figure(data):
    latest = data.iloc[-1]
    kpi_specs = [("SPY Drawdown", latest["SPY_drawdown"] * 100, "number", "%"), ("Breadth > 200D", latest["Breadth_above_200D"], "%"), ("HY OAS", latest["HY_OAS"], " pts"), ("VIX Level", latest["^VIX"], ""), ("TNX Mom Z", latest["TNX_momentum_z"], ""), ("Risk Score", latest["Composite_risk_score"], "%")]
    figure = make_subplots(rows=2, cols=3, specs=[[{"type": "indicator"}] * 3, [{"type": "indicator"}] * 3], vertical_spacing=0.15)
    for index, item in enumerate(kpi_specs):
        title, value, suffix = item if len(item) == 3 else (item[0], item[1], item[3])
        row, col = index // 3 + 1, index % 3 + 1
        trace = go.Indicator(mode="gauge+number" if title == "Risk Score" else "number", value=value, title={"text": title}, number={"suffix": suffix}, gauge={"axis": {"range": [0, 100]}, "bar": {"thickness": 0.35}, "steps": [{"range": [0, 30], "color": "#1f7a1f"}, {"range": [30, 60], "color": "#b58900"}, {"range": [60, 100], "color": "#8b0000"}]} if title == "Risk Score" else None)
        figure.add_trace(trace, row=row, col=col)
    figure.update_layout(height=520, template="plotly_dark", margin=dict(l=20, r=20, t=50, b=20), title="Top Market Dashboard")
    return figure

def make_market_figure(data):
    chart = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.42, 0.18, 0.18, 0.22], specs=[[{"secondary_y": True}], [{"secondary_y": False}], [{"secondary_y": False}], [{"secondary_y": False}]])
    chart.add_trace(go.Scatter(x=data.index, y=data[MAIN_TICKER], name="SPY", line=dict(width=2)), row=1, col=1, secondary_y=False)
    chart.add_trace(go.Scatter(x=data.index, y=data["SPY_200D"], name="SPY 200D", line=dict(width=1, dash="dash")), row=1, col=1, secondary_y=False)
    chart.add_trace(go.Scatter(x=data.index, y=data["Composite_risk_score"], name="Risk Score", line=dict(width=2)), row=1, col=1, secondary_y=True)
    chart.add_trace(go.Scatter(x=data.index, y=data["^VIX"], name="VIX Level", line=dict(width=2)), row=2, col=1)
    chart.add_trace(go.Scatter(x=data.index, y=data["HY_OAS"], name="HY OAS", line=dict(width=2)), row=3, col=1)
    chart.add_trace(go.Scatter(x=data.index, y=data["TNX_momentum_z"], name="TNX Momentum Z", line=dict(width=2)), row=4, col=1)
    chart.add_trace(go.Scatter(x=data.index, y=np.ones(len(data)), name="+1 Z", line=dict(width=1, dash="dash")), row=4, col=1)
    chart.add_trace(go.Scatter(x=data.index, y=-np.ones(len(data)), name="-1 Z", line=dict(width=1, dash="dash")), row=4, col=1)
    chart.update_layout(height=1150, template="plotly_dark", margin=dict(l=20, r=20, t=50, b=20), title="Market State Panel")
    chart.update_yaxes(title_text="SPY", row=1, col=1, secondary_y=False)
    chart.update_yaxes(title_text="Risk Score", row=1, col=1, secondary_y=True, range=[0, 100])
    chart.update_yaxes(title_text="VIX", row=2, col=1)
    chart.update_yaxes(title_text="HY OAS", row=3, col=1)
    chart.update_yaxes(title_text="TNX Z", row=4, col=1)
    return chart

def make_term_structure_figure(data):
    figure = make_subplots(rows=1, cols=2, subplot_titles=("VIX Term Structure - Last 10 Days", "Yield Term Structure - Last 10 Days"))
    vix_dates = data[["^VIX", "^VIX3M", "^VIX6M"]].dropna().tail(10)
    curve_dates = data[TREASURY_ORDER].dropna().tail(10)
    for date, row in vix_dates.iterrows():
        figure.add_trace(go.Scatter(x=["Spot VIX", "3M VIX", "6M VIX"], y=[row["^VIX"], row["^VIX3M"], row["^VIX6M"]], mode="lines+markers", name=date.strftime("%Y-%m-%d"), legendgroup=date.strftime("%Y-%m-%d")), row=1, col=1)
    for date, row in curve_dates.iterrows():
        figure.add_trace(go.Scatter(x=TREASURY_ORDER, y=row.values, mode="lines+markers", name=date.strftime("%Y-%m-%d"), legendgroup=date.strftime("%Y-%m-%d"), showlegend=False), row=1, col=2)
    figure.update_layout(height=500, template="plotly_dark", margin=dict(l=20, r=20, t=60, b=20), title="Term Structure Panel", legend_title="Date")
    figure.update_yaxes(title_text="VIX", row=1, col=1)
    figure.update_yaxes(title_text="Yield (%)", row=1, col=2)
    return figure

def style_summary_table(sector_return_table):
    table = sector_return_table.copy()
    table["Above_200D"] = table["Above_200D"].map({1: "Yes", 0: "No"})
    return table.style.format({"1D": "{:.2f}", "1W": "{:.2f}", "1M": "{:.2f}", "3M": "{:.2f}"}).background_gradient(subset=["1D", "1W", "1M", "3M"], cmap="RdYlGn")


In [7]:
def run_dashboard(start_date="2010-01-01"):
    data, risk_components, sector_return_table = compute_dashboard_data(start_date)
    display(HTML("<h3>1) KPI Cards</h3>"))
    make_kpi_figure(data).show()
    display(HTML("<h3>2) Market State Panel</h3>"))
    make_market_figure(data).show()
    display(HTML("<h3>3) Term Structure Panel</h3>"))
    make_term_structure_figure(data).show()
    display(HTML("<h3>4) Sector Summary Table</h3>"))
    display(style_summary_table(sector_return_table))
    latest = data.iloc[-1]
    hy_oas_threshold = data["HY_OAS"].rolling(252).quantile(0.75).iloc[-1]
    summary = pd.DataFrame({
        "Metric": ["SPY vs 200D", "Breadth > 200D", "HY OAS", "SPY Drawdown", "VIX Level", "VIX Contango", "TNX Momentum Z", "Composite Risk Score"],
        "Value": ["Above" if latest[MAIN_TICKER] > latest["SPY_200D"] else "Below", f"{latest['Breadth_above_200D']:.1f}%", f"{latest['HY_OAS']:.2f}", f"{latest['SPY_drawdown'] * 100:.2f}%", f"{latest['^VIX']:.2f}", f"{latest['VIX_contango_pct']:.2f}%", f"{latest['TNX_momentum_z']:.2f}", f"{latest['Composite_risk_score']:.1f}"],
        "Healthy": ["Yes" if latest[MAIN_TICKER] > latest["SPY_200D"] else "No", "Yes" if latest["Breadth_above_200D"] >= 45 else "No", "Yes" if latest["HY_OAS"] <= hy_oas_threshold else "No", "Yes" if latest["SPY_drawdown"] >= -0.08 else "No", "Yes" if latest["^VIX"] <= 25 else "No", "Yes" if latest["VIX_contango_pct"] >= 0 else "No", "Yes" if abs(latest["TNX_momentum_z"]) <= 1 else "No", "Yes" if latest["Composite_risk_score"] < 50 else "No"]
    }, index=range(1, 9))
    display(HTML("<h3>5) Quick Read</h3>"))
    display(summary)
    return data, risk_components, sector_return_table


In [ ]:
# Run the full dashboard
data, risk_components, sector_return_table = run_dashboard(start_date="2010-01-01")

,1D,1W,1M,3M,Above_200D
XLE,0.00,1.60,6.66,34.51,Yes
XLU,0.00,0.49,2.46,9.39,Yes
XLC,0.00,-1.40,-0.81,-2.25,Yes
XLK,0.00,0.09,-2.09,-3.08,Yes
XLRE,0.00,-0.28,-3.07,4.76,Yes
XLY,0.00,-0.85,-5.51,-9.14,No
XLI,0.00,-0.04,-5.63,6.21,Yes
XLP,0.00,-1.91,-6.13,5.07,Yes
XLV,0.00,-2.01,-6.68,-4.09,Yes
XLF,0.00,0.29,-6.88,-10.06,No


,Metric,Value,Healthy
1,SPY vs 200D,Above,Yes
2,Breadth > 200D,81.8%,Yes
3,HY OAS,3.22,No
4,SPY Drawdown,-4.90%,Yes
5,VIX Level,26.16,No
6,VIX Contango,1.53%,Yes
7,TNX Momentum Z,1.50,No
8,Composite Risk Score,37.5,Yes



## Risk score logic

The composite risk score is the average of 8 binary flags:

1. SPY below 200D
2. Breadth above 200D below 45%
3. HY OAS above its rolling 75th percentile
4. SPY drawdown below -8%
5. VIX above 20
6. VIX contango below 0
7. TNX momentum z-score outside -1 to +1


## Additional checks if in recession (Inflation/Deflation)

1. Macroeconomic data (Fed Fund Rate decision, Jobless claim/Unemployment rate, GDP/CPI/PPI/PCE)
2. Cross sectional Assests (Gold, Oil, Commodities)
3. Openinsider Signal
